In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tensorflow as tf

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import Healpy_ViT, Healpy_Transformer, HealpyPseudoConv

from msfm.utils import files

from deep_lss.models.delta_model import DeltaLossModel
from deep_lss.nets.resnet import ResNetLayers
# from deep_lss.models.grid_model import GridLossModel
# from msfm.fiducial_pipeline import FiducialPipeline

24-02-14 00:54:41   imports.py INF   Setting up healpy to run on 32 CPUs 


### constants

In [3]:
n_side = 512
data_vec_pix, _, _, _ = files.load_pixel_file()
n_z_bins = 8
n_second_to_last_features = 128
n_output = 7

24-02-14 00:54:42     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_512.h5 


### random batch

In [4]:
# for delta loss
local_batch_size = 4 * (2 * n_output + 1)
batch = tf.random.uniform((local_batch_size, len(data_vec_pix), n_z_bins), dtype=tf.float32)

# ResNet baseline

In [5]:
layers = ResNetLayers(
    n_base_channels=32,
    n_downsampling=3,
    n_cheby=2,
    n_residuals=5,
    poly_degree=5,
    n_second_to_last_features=n_second_to_last_features,
).get_layers()

resnet_model = DeltaLossModel(
    network=layers,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=20,
    input_shape=(None, len(data_vec_pix), n_z_bins),
)

24-02-14 00:54:44 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv (HealpyP  (None, 116224, 32)       1056      
 seudoConv)                                                      
                                                                 
 healpy_pseudo_conv_1 (Healp  (None, 29056, 64)        8256      
 yPseudoConv)                                                    
                                                                 
 healpy_pseudo_conv_2 (Healp  (None, 7264, 128)        32896     
 yPseudoConv)                                                    
                                                               

In [6]:
%%timeit
resnet_model(batch)

184 ms ± 214 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
layers = [
    Healpy_ViT(p=4, key_dim=128, num_heads=4, positional_encoding=True, n_layers=4, layer_norm=True),
    tf.keras.layers.Flatten(),
    tf.keras.layers.LayerNormalization(axis=-1),
    # tf.keras.layers.Dense(n_second_to_last_channels, activation="relu"),
    # tf.keras.layers.Dropout(0.0),
    tf.keras.layers.Dense(n_output),
]

vit_model = DeltaLossModel(
    network=layers,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=20,
    input_shape=(None, len(data_vec_pix), n_z_bins),
)

24-02-14 00:54:49 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 16.0, the input with nside 512 will be transformed to 32 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy__vi_t (Healpy_ViT)   (None, 1816, 512)         6189568   
                                                                 
 flatten_1 (Flatten)         (None, 929792)            0         
                                                                 
 layer_normalization_21 (Lay  (None, 929792)           1859584   
 erNormalization)                                                
                                                                 
 dense_18 (Dense)            (None, 7)                 6508551   
                                                               

In [8]:
%%timeit
vit_model(batch)

130 ms ± 263 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


# 1D Convolutions

In [9]:
# class ResidualBlock(tf.keras.Model):
#     def __init__(self, layer1, layer2, use_norm=True, norm_kwargs={}, norm_type="layer_norm", name=""):
#         super(ResidualBlock, self).__init__(name=name)
        
#         self.layer1 = layer1
#         self.layer2 = layer2

#         if use_norm:
#             if norm_type=="layer_norm":
#                 self.norm1 = tf.keras.layers.LayerNormalization(**norm_kwargs)
#                 self.norm2 = tf.keras.layers.LayerNormalization(**norm_kwargs)
#             elif norm_type=="batch_norm":
#                 self.norm1 = tf.keras.layers.BatchNormalization(**norm_kwargs)
#                 self.norm2 = tf.keras.layers.BatchNormalization(**norm_kwargs)
#             else:
#                 raise NotImplementedError
#         else:
#             raise NotImplementedError

#     def call(self, input_tensor, training=False):
#         x = self.layer1(input_tensor)
#         x = self.norm1(x, training=training)

#         x = self.layer2(input_tensor)
#         x = self.norm2(x, training=training)

#         x += input_tensor

#         return tf.nn.relu(x)


In [10]:
# n_base_channels = 8
# n_downsampling = 5
# n_residuals = 5

# kernel_size = 9
# activation = "relu"

# layers = []

# n_channels = n_base_channels
# for i in range(n_downsampling):
#     layers.append(tf.keras.layers.Conv1D(n_channels, kernel_size, padding="same", activation=activation, name=f"1d_conv_{i}"))
#     layers.append(tf.keras.layers.LayerNormalization(axis=-1, name=f"layer_norm_{i}"))
    
#     n_channels *= 2
#     layers.append(HealpyPseudoConv(p=1, Fout=n_channels, activation=activation, name=f"pseudo_conv_{i}"))


# for i in range(n_residuals):
#     layers.append(
#         ResidualBlock(
#             layer1=tf.keras.layers.Conv1D(n_channels, kernel_size, padding="same", activation=activation),
#             layer2=tf.keras.layers.Conv1D(n_channels, kernel_size, padding="same", activation=activation),
#             use_norm=True,
#             norm_type="layer_norm",
#             name=f"residual_1d_conv_{i}",
#         )
#     )
            
# layers.append(tf.keras.layers.Flatten())
# layers.append(tf.keras.layers.LayerNormalization(axis=-1))
# layers.append(tf.keras.layers.Dense(n_second_to_last_channels, activation="relu"))
# layers.append(tf.keras.layers.Dropout(0.0))
# layers.append(tf.keras.layers.Dense(n_output))

# conv_model = DeltaLossModel(
#     network=layers,
#     n_side=n_side,
#     indices=data_vec_pix,
#     n_neighbors=20,
#     input_shape=(None, len(data_vec_pix), n_z_bins),
# )

In [11]:
# %%timeit
# conv_model(batch)

In [12]:
from deep_lss.nets.one_d_conv import OneDConvLayers

In [13]:
layers = OneDConvLayers(
    out_dim=n_output,
    # n_base_channels=1,
    # n_downsampling=0,
    # n_residuals=1,
    # n_second_to_last_features=None,
    # # n_residuals=0,
).get_layers()

conv_model = DeltaLossModel(
    network=layers,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=20,
    input_shape=(None, len(data_vec_pix), n_z_bins),
)

24-02-14 00:54:53 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 1d_conv_0 (Conv1D)          (None, 464896, 8)         584       
                                                                 
 layer_norm_0 (LayerNormaliz  (None, 464896, 8)        16        
 ation)                                                          
                                                                 
 healpy_pseudo_conv_5 (Healp  (None, 116224, 16)       528       
 yPseudoConv)                                                    
                                                                 
 1d_conv_1 (Conv1D)          (None, 116224, 16)        2320    

In [14]:
%%timeit
conv_model(batch)

The slowest run took 14.81 times longer than the fastest. This could mean that an intermediate result is being cached.
237 ms ± 183 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
# %%time
# trainable_variables = conv_model.network.trainable_variables

# with tf.GradientTape() as tape:
#     predictions = conv_model(batch)
#     loss = tf.math.reduce_mean(predictions)
    
# gradients = tape.gradient(loss, trainable_variables)

# print(gradients)

In [16]:
gradients

NameError: name 'gradients' is not defined